## Bases vectorielles — construction et extension

Construit les bases vectorielles SigLIP (768D, niveaux de gris) utilisées par
le RAG (`rag_generation.ipynb`, `app_rag.py`) : une base Bibles, une base
Ovide (73 illustrations thématisées `BNU_corpus.ods`, aussi persistée pour
`../comparaison_ovide_bibles/`), puis une base Ovide complète (2191
illustrations, 28 éditions, métadonnées `Synthese`).

Le choix du modèle d'embedding (SigLIP gelé plutôt que CLIP/DINOv2/SigLIP
fine-tuné) a été validé séparément dans `selection_modele.ipynb` — ce
notebook recalcule ses propres embeddings via `rag_utils.charger_siglip` /
`embed_image`, pour rester exécutable seul, avec le même code que celui
utilisé au moment de la requête (retrieval).

In [1]:
from pathlib import Path

import re
import numpy as np
import pandas as pd
import requests
import torch

from rag_utils import charger_siglip, embed_image

RACINE = Path("../../").resolve()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)


Device : cuda


## Bases vectorielles — Bibles + Ovide (corpus BNU·Céline)

Objectif : construire deux bases vectorielles persistantes et comparables
(SigLIP, niveaux de gris) — l'une pour les Bibles (déjà classées), l'autre
pour les illustrations d'Ovide listées par Céline dans
`retours_celine/BNU_corpus.ods` (3 feuilles : `création_1` = création du
monde, `création_2` = création de l'homme, `Déluge`).

### 1. Images Bibles et embeddings SigLIP

Dataset complet des Bibles déjà classées par thème (`classes_celine`), à
l'exclusion des classes non iconographiques (`Images CURIEUSES`,
`Images MIXTES`) — même filtre que dans `selection_modele.ipynb`.

In [2]:
processor, model_siglip = charger_siglip(DEVICE)
print("SigLIP charge")

CLASSES_DIR = RACINE / "data" / "bibles_mdz" / "classes_celine"
THEMES_EXCLUS = {"Images CURIEUSES", "Images MIXTES"}

bible_images = []  # liste de dicts : chemin, theme
for dossier in sorted(CLASSES_DIR.iterdir()):
    if not dossier.is_dir() or dossier.name in THEMES_EXCLUS:
        continue
    fichiers = [f for f in dossier.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
    for f in fichiers:
        bible_images.append({"chemin": f, "theme": dossier.name})

themes_bible = [i["theme"] for i in bible_images]
print(f"{len(bible_images)} images Bible sur {len(set(themes_bible))} themes")

chemins_bible = [i["chemin"] for i in bible_images]
print("Embeddings SigLIP (Bibles)...")
X_siglip_bible = np.array([
    embed_image(chemin, processor, model_siglip, DEVICE) for chemin in chemins_bible
])
print(X_siglip_bible.shape)


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

SigLIP charge
372 images Bible sur 12 themes
Embeddings SigLIP (Bibles)...


(372, 768)


### 2. Tableau unifié BNU_corpus (3 feuilles -> format long)

In [3]:
import pandas as pd
import requests

def nettoyer_feuille(df, theme):
    df = df.copy()
    df = df[df['titre'].notna()]
    df = df[df['titre'].astype(str).str.strip().str.lower() != 'titre']  # ligne d'en-tete parasite
    for col in ['url_1', 'url_2']:
        if col in df.columns:
            df[col] = df[col].replace({'absent': np.nan, 'Absent': np.nan})
    df['theme'] = theme
    return df

COLS_20 = ['url_1','url_2','url_catalogue','url_gallica','type','titre','ville','publisher','annee','langue',
           'format','nb_pages','nb_vues_num','nb_gravures','technique','graveur','taille_gravures',
           'type_iconographique','cadre','autres']

CHEMIN_BNU_CORPUS = RACINE / "retours_celine" / "BNU_corpus.ods"

c1 = pd.read_excel(CHEMIN_BNU_CORPUS, engine='odf', sheet_name='création_1', header=None)
c1.columns = COLS_20
c1 = nettoyer_feuille(c1, 'creation_monde')

c2 = pd.read_excel(CHEMIN_BNU_CORPUS, engine='odf', sheet_name='création_2')
c2.columns = ['url_1','url_catalogue','url_gallica','type','titre','ville','publisher','annee','langue',
              'format','nb_pages','nb_vues_num','nb_gravures','technique','graveur','taille_gravures',
              'type_iconographique','cadre','autres']
c2['url_2'] = np.nan
c2 = nettoyer_feuille(c2, 'creation_homme')

dl = pd.read_excel(CHEMIN_BNU_CORPUS, engine='odf', sheet_name='Déluge')
dl.columns = COLS_20
dl = nettoyer_feuille(dl, 'deluge')

bnu_corpus = pd.concat([c1, c2, dl], ignore_index=True)
bnu_corpus['annee'] = pd.to_numeric(bnu_corpus['annee'], errors='coerce')
bnu_corpus['variante'] = 'principale'

# url_2, quand renseignee, pointe vers une AUTRE illustration de la meme edition
# et du meme theme (verifie : jamais identique a url_1) - on l'ajoute comme ligne
# supplementaire plutot que de l'ignorer.
url2_valides = bnu_corpus[
    bnu_corpus['url_2'].notna() & (bnu_corpus['url_2'].astype(str) != bnu_corpus['url_1'].astype(str))
].copy()
url2_valides['url_1'] = url2_valides['url_2']
url2_valides['variante'] = 'secondaire'

bnu_corpus = pd.concat([bnu_corpus, url2_valides], ignore_index=True)
print(f"{len(bnu_corpus)} lignes ({len(url2_valides)} variantes url_2 ajoutees)")
print(bnu_corpus['theme'].value_counts())

95 lignes (18 variantes url_2 ajoutees)
theme
deluge            38
creation_monde    30
creation_homme    27
Name: count, dtype: int64


### 3. Résolution des images

Stratégie en deux temps :
1. **Réutiliser l'existant** — 20 des éditions listées dans BNU_corpus ont déjà
   été téléchargées et segmentées par ailleurs dans ce projet (corpus de 28
   éditions dans `data/editions_ovide/segmentees/`, construit pour la
   classification bois/cuivre). On y cherche directement la page ciblée avant
   de retélécharger quoi que ce soit.
2. **Téléchargement + découpe YOLO** — pour les pages non couvertes par
   l'existant, on télécharge la page via l'API IIIF (Gallica ou
   digitale-sammlungen) et on applique le même détecteur YOLO que pour le
   reste du corpus Ovide (`gallica_utils.segmenter_page`), pour rester
   méthodologiquement cohérent avec le corpus déjà construit.

In [4]:
import os

# éditions BNU_corpus déjà téléchargées + segmentées ailleurs dans le projet
# (classification_graveur/01_constitution_dataset.ipynb, classification_bois_cuivre/01_dataset.ipynb)
ID_VERS_DOSSIER = {
    'btv1b2200047r':  'bois_salomon_rouille_lyon1557',
    'bsb10139926':    'bois_wickram_behem_mayence1545',
    'bsb00087854':    'bois_solis_feyerabend_francfort1581',
    'bsb10863401':    'cuivre_savery_farnaby_paris1637',
    'btv1b22000559':  'bois_eskrich_rouille_lyon1556',
    'bsb11054210':    'bois_leroy_gueynard_lyon1510',
    'bsb10872075':    'cuivre_baur_sn_augsbourg1709',
    'bsb10872073':    'cuivre_baur_sn_vienne1639',
    'bsb00004340':    'cuivre_borcht_plantin_anvers1591',
    'bpt6k15151988':  'cuivre_bouche_blaeu_amsterdam1702',
    'bpt6k15218623':  'cuivre_depasse_depasse_koln1602',
    'bpt6k1522448r':  'cuivre_depasse_jansonius_arnhem1607',
    'bsb11913284':    'cuivre_gaultier_guillemot_paris1610',
    'bpt6k6277348n':  'cuivre_gaultier_veuveguillemot_paris1614',
    'btv1b10534945n': 'cuivre_goltzius_goltzius_haarlem1589',
    'bpt6k722055':    'cuivre_isaac_langelier_paris1617',
    'btv1b22000826':  'cuivre_mathieu_langelier_paris1619',
    'bpt6k87045023':  'cuivre_monconet_sommaville_paris1660',
    'btv1b54000051z': 'cuivre_tempesta_dejode_anvers1606',
    'bsb00008186':    'cuivre_tempesta_jansonius_amsterdam1610',
}

SEG_DIR = RACINE / "data" / "editions_ovide" / "segmentees"

def indexer_dossier(dossier):
    """Indexe {page: [(confiance, nom_fichier), ...]} pour les crops déjà présents."""
    idx = {}
    for f in os.listdir(SEG_DIR / dossier):
        m = re.search(r'0*(\d+)_det(\d+)_conf([\d.]+)\.jpg$', f)
        if m:
            page, conf = int(m.group(1)), float(m.group(3))
            idx.setdefault(page, []).append((conf, f))
    return idx

INDEX_LOCAL = {d: indexer_dossier(d) for d in set(ID_VERS_DOSSIER.values())}

PAT_GALLICA = re.compile(r'ark:/12148/([a-zA-Z0-9]+)/f(\d+)')
PAT_MDZ = re.compile(r'digitale-sammlungen\.de/en/view/(bsb\d+)\?page=(\d+)')

def resoudre_source(url):
    """Retourne (statut, source, identifiant, page, chemin_local_existant|None)."""
    if pd.isna(url):
        return 'sans_url', None, None, None, None
    url = str(url)
    m = PAT_GALLICA.search(url)
    source = 'gallica'
    if not m:
        m = PAT_MDZ.search(url)
        source = 'mdz'
    if not m:
        return 'ignore', None, None, None, None  # hathitrust ou format non gere

    ident, page = m.group(1), int(m.group(2))
    dossier = ID_VERS_DOSSIER.get(ident)
    if dossier:
        idx = INDEX_LOCAL[dossier]
        if page in idx:
            conf, fichier = max(idx[page], key=lambda t: t[0])
            return 'local', source, ident, page, str(SEG_DIR / dossier / fichier)
        proches = [p for p in idx if abs(p - page) <= 2]
        if proches:
            p_best = min(proches, key=lambda p: abs(p - page))
            conf, fichier = max(idx[p_best], key=lambda t: t[0])
            return 'local', source, ident, page, str(SEG_DIR / dossier / fichier)

    return 'a_telecharger', source, ident, page, None

resolution = bnu_corpus['url_1'].apply(resoudre_source)
bnu_corpus[['statut', 'source', 'ident', 'page', 'chemin_local']] = pd.DataFrame(
    resolution.tolist(), index=bnu_corpus.index
)
print(bnu_corpus['statut'].value_counts())
print(f"\n{(bnu_corpus['statut'] == 'local').sum()} pages déjà disponibles localement (corpus déjà segmenté)")

statut
a_telecharger    70
local            21
sans_url          3
ignore            1
Name: count, dtype: int64

21 pages déjà disponibles localement (corpus déjà segmenté)


Téléchargement + découpe YOLO pour les pages non couvertes par l'existant.

In [5]:
import sys
import time

sys.path.append(str(RACINE / "notebooks"))
from gallica_utils import charger_yolo, segmenter_page, liberer_yolo

DOSSIER_PAGES_BNU = RACINE / "data" / "editions_ovide" / "pages_brutes" / "bnu_corpus_celine"
DOSSIER_CROPS_BNU = RACINE / "data" / "editions_ovide" / "segmentees" / "bnu_corpus_celine"
DOSSIER_PAGES_BNU.mkdir(parents=True, exist_ok=True)
DOSSIER_CROPS_BNU.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Gecko/20100101 Firefox/128.0"}

def url_page_brute(source, ident, page):
    if source == 'gallica':
        return f"https://gallica.bnf.fr/iiif/ark:/12148/{ident}/f{page}/full/full/0/native.jpg"
    return f"https://api.digitale-sammlungen.de/iiif/image/v2/{ident}_{page:05d}/full/full/0/default.jpg"

def telecharger_page(source, ident, page, max_essais=4, pause_base=3):
    """Téléchargement avec pause entre requêtes + repli exponentiel sur erreur
    réseau/429 (Gallica bloque les rafales de requêtes sans délai)."""
    page = int(page)
    chemin_page = DOSSIER_PAGES_BNU / f"{ident}_f{page:03d}.jpg"
    if chemin_page.exists():
        return chemin_page, None

    derniere_erreur = None
    for essai in range(max_essais):
        try:
            r = requests.get(url_page_brute(source, ident, page), headers=HEADERS, timeout=30)
            if r.status_code == 429:
                attente = pause_base * (2 ** essai)
                derniere_erreur = f"429 persistant (essai {essai + 1})"
                time.sleep(attente)
                continue
            r.raise_for_status()
            chemin_page.write_bytes(r.content)
            return chemin_page, None
        except Exception as e:
            derniere_erreur = str(e)
            time.sleep(pause_base * (2 ** essai))
    return None, derniere_erreur

a_telecharger = bnu_corpus[bnu_corpus['statut'].isin(['a_telecharger', 'erreur_telechargement', 'aucune_illustration_detectee'])]
print(f"{len(a_telecharger)} pages à télécharger + segmenter")

modele_yolo = charger_yolo(str(RACINE / "yolov5_repo"))

resultats_dl = {}
erreurs_detail = {}
for i, (idx, row) in enumerate(a_telecharger.iterrows(), 1):
    print(f"  {i}/{len(a_telecharger)}...", end="\r")
    cle = (row['source'], row['ident'], row['page'])
    if cle in resultats_dl:
        continue
    chemin_page, erreur = telecharger_page(*cle)
    time.sleep(1.0 if row['source'] == 'gallica' else 0.2)  # pacing anti rate-limit
    if erreur:
        resultats_dl[cle] = ('erreur_telechargement', None)
        erreurs_detail[cle] = erreur
        continue
    prefixe = f"{row['ident']}_f{int(row['page']):03d}"
    nb = segmenter_page(str(chemin_page), prefixe, str(DOSSIER_CROPS_BNU), modele_yolo, conf_thres=0.25)
    if nb == 0:
        resultats_dl[cle] = ('aucune_illustration_detectee', None)
        continue
    crops = sorted(DOSSIER_CROPS_BNU.glob(f"{prefixe}_det*_conf*.jpg"))
    meilleur = max(crops, key=lambda p: float(re.search(r'conf([\d.]+)\.jpg$', p.name).group(1)))
    resultats_dl[cle] = ('telecharge', str(meilleur))
print()

liberer_yolo(modele_yolo)

for idx, row in a_telecharger.iterrows():
    statut, chemin = resultats_dl[(row['source'], row['ident'], row['page'])]
    bnu_corpus.loc[idx, 'statut'] = statut
    if chemin:
        bnu_corpus.loc[idx, 'chemin_local'] = chemin

print(bnu_corpus['statut'].value_counts())
if erreurs_detail:
    print("\nExemples d'erreurs :")
    for cle, msg in list(erreurs_detail.items())[:5]:
        print(f"  {cle} : {msg}")

70 pages à télécharger + segmenter


✓ YOLO chargé — classes : {0: 'illustration'}


✓ Mémoire GPU libérée
statut
telecharge                      61
local                           21
aucune_illustration_detectee     9
sans_url                         3
ignore                           1
Name: count, dtype: int64


### 4. Vectorisation SigLIP (niveaux de gris) des illustrations Ovide résolues

In [6]:
ovide_bnu_resolus = bnu_corpus[bnu_corpus['chemin_local'].notna()].copy()
avant_dedup = len(ovide_bnu_resolus)
ovide_bnu_resolus = ovide_bnu_resolus.drop_duplicates(subset='chemin_local').reset_index(drop=True)
print(f"{len(ovide_bnu_resolus)} illustrations Ovide (BNU_corpus) résolues sur {len(bnu_corpus)} lignes"
      f" ({avant_dedup - len(ovide_bnu_resolus)} doublons de fichier retires)")
print(ovide_bnu_resolus['theme'].value_counts())
print(ovide_bnu_resolus['variante'].value_counts())

chemins_ovide_bnu = [Path(p) for p in ovide_bnu_resolus['chemin_local']]
print("\nEmbeddings SigLIP (Ovide BNU_corpus)...")
X_siglip_ovide_bnu = np.array([embed_image(chemin, processor, model_siglip, DEVICE) for chemin in chemins_ovide_bnu])
print(X_siglip_ovide_bnu.shape)

73 illustrations Ovide (BNU_corpus) résolues sur 95 lignes (9 doublons de fichier retires)
theme
deluge            31
creation_monde    25
creation_homme    17
Name: count, dtype: int64
variante
principale    57
secondaire    16
Name: count, dtype: int64

Embeddings SigLIP (Ovide BNU_corpus)...


(73, 768)


### 5. Sauvegarde des deux bases vectorielles


In [7]:
DOSSIER_VECTOR_DB = RACINE / "data" / "vector_bases"
DOSSIER_VECTOR_DB.mkdir(exist_ok=True)

# URLs distantes (Gallica / digitale-sammlungen) plutôt que des chemins locaux,
# pour que le fichier reste utilisable si on le partage
def urls_bsb(bsb_id, page):
    page = int(page)
    return (
        f"https://www.digitale-sammlungen.de/en/view/{bsb_id}?page={page}",
        f"https://api.digitale-sammlungen.de/iiif/image/v2/{bsb_id}_{page:05d}/full/full/0/default.jpg",
    )

def urls_gallica(ark, folio):
    folio = int(folio)
    return (
        f"https://gallica.bnf.fr/ark:/12148/{ark}/f{folio}.item",
        f"https://gallica.bnf.fr/iiif/ark:/12148/{ark}/f{folio}/full/full/0/native.jpg",
    )

# Base vectorielle Bibles
base_bibles = pd.DataFrame({
    "chemin": [str(i["chemin"]) for i in bible_images],
    "theme": themes_bible,
})
base_bibles["embedding"] = [v.tolist() for v in X_siglip_bible]

# le bsb_id et le numero de page sont toujours presents dans le nom de fichier
# (ex: bsb00085642_page031_det1_conf0.92.jpg) - tout le corpus Bibles vient de
# digitale-sammlungen (dossier data/bibles_mdz/).
PAT_BIBLE = re.compile(r'^(bsb\d+)_page(\d+)_det')
url_page_b, url_image_b = [], []
for chemin in base_bibles["chemin"]:
    m = PAT_BIBLE.match(Path(chemin).name)
    up, ui = urls_bsb(m.group(1), m.group(2)) if m else (None, None)
    url_page_b.append(up)
    url_image_b.append(ui)
base_bibles["url_page"] = url_page_b
base_bibles["url_image"] = url_image_b

base_bibles.to_pickle(DOSSIER_VECTOR_DB / "bibles_siglip.pkl")
print(f"Base vectorielle Bibles : {base_bibles.shape} -> {DOSSIER_VECTOR_DB / 'bibles_siglip.pkl'}")
print(f"  url_page manquante : {base_bibles['url_page'].isna().sum()} / {len(base_bibles)}")

# Base vectorielle Ovide (corpus BNU . Céline)
base_ovide = ovide_bnu_resolus[[
    "chemin_local", "theme", "titre", "ville", "publisher", "annee", "langue", "graveur", "technique", "variante",
    "ident", "page", "source"
]].rename(columns={"chemin_local": "chemin", "source": "source_plateforme"})
base_ovide["embedding"] = [v.tolist() for v in X_siglip_ovide_bnu]

url_page_o, url_image_o = [], []
for _, r in base_ovide.iterrows():
    if r["source_plateforme"] == "gallica":
        up, ui = urls_gallica(r["ident"], r["page"])
    else:
        up, ui = urls_bsb(r["ident"], r["page"])
    url_page_o.append(up)
    url_image_o.append(ui)
base_ovide["url_page"] = url_page_o
base_ovide["url_image"] = url_image_o

chemin_ovide_comparaison = DOSSIER_VECTOR_DB / "ovide_bnu_corpus_comparaison_siglip.pkl"
base_ovide.to_pickle(chemin_ovide_comparaison)
print(f"Base vectorielle Ovide (73 illustrations) : {base_ovide.shape} -> {chemin_ovide_comparaison}")
print("Persistee uniquement pour ../comparaison_ovide_bibles/ (pas utilisee par le RAG,"
      " qui repose sur la base complete 2191 illustrations ci-dessous).")

Base vectorielle Bibles : (372, 5) -> /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/vector_bases/bibles_siglip.pkl
  url_page manquante : 0 / 372
Base vectorielle Ovide (73 illustrations) : (73, 16) -> /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/vector_bases/ovide_bnu_corpus_comparaison_siglip.pkl
Persistee uniquement pour ../comparaison_ovide_bibles/ (pas utilisee par le RAG, qui repose sur la base complete 2191 illustrations ci-dessous).


## Base vectorielle complete — 28 editions segmentees, metadonnees Synthese

Objectif : etendre la base vectorielle Ovide au-dela des 73 illustrations deja
themees (construites ci-dessus, en memoire, a partir de 3 feuilles de
`BNU_corpus.ods`). Ici on vectorise **toutes** les
illustrations deja segmentees dans `data/editions_ovide/segmentees/` (2191 crops,
28 dossiers/editions), chacune enrichie avec les metadonnees d'edition trouvees
dans la feuille **Synthese** de `BNU_corpus.ods` (titre, ville, graveur, technique,
annee, langue, famille iconographique).

**Ce que ça n'est pas** : un theme (deluge / creation_monde / ...) precis par
illustration — sauf pour les 73 deja connues, reprises telles quelles ici. Les
~2100 autres n'ont pas de theme individuel (voir feuilles `#N` de `BNU_corpus.ods`
pour une etape ulterieure, qui donnera un theme precis pour les editions ayant une
feuille dediee — Salomon, Wickram, Solis, Savery notamment).

In [8]:
import unicodedata  # normalisation accents/casse pour le rattachement des theme (voir plus bas)


### 1. Rapprochement dossier segmente -> ligne Synthese

Deux voies : ark deja connu (documente dans les notebooks
`classification_bois_cuivre` / `classification_graveur`), ou a defaut un
rapprochement par ville + annee + fragment du nom du graveur (verifie a l'oeil,
voir le tableau affiche plus bas — 27/27 dossiers rapproches avec succes).

In [9]:
SEG_DIR = RACINE / "data" / "editions_ovide" / "segmentees"

# ark deja connus avec certitude (repris de vector_base.ipynb / classification_*/01_*.ipynb)
ARK_CONNU = {
    'bois_salomon_rouille_lyon1557': 'btv1b2200047r',
    'bois_wickram_behem_mayence1545': 'bsb10139926',
    'bois_solis_feyerabend_francfort1581': 'bsb00087854',
    'cuivre_savery_farnaby_paris1637': 'bsb10863401',
    'bois_eskrich_rouille_lyon1556': 'btv1b22000559',
    'bois_leroy_gueynard_lyon1510': 'bsb11054210',
    'cuivre_baur_sn_augsbourg1709': 'bsb10872075',
    'cuivre_baur_sn_vienne1639': 'bsb10872073',
    'cuivre_borcht_plantin_anvers1591': 'bsb00004340',
    'cuivre_bouche_blaeu_amsterdam1702': 'bpt6k15151988',
    'cuivre_depasse_depasse_koln1602': 'bpt6k15218623',
    'cuivre_depasse_jansonius_arnhem1607': 'bpt6k1522448r',
    'cuivre_gaultier_guillemot_paris1610': 'bsb11913284',
    'cuivre_gaultier_veuveguillemot_paris1614': 'bpt6k6277348n',
    'cuivre_goltzius_goltzius_haarlem1589': 'btv1b10534945n',
    'cuivre_goltzius_goltzius_haarlem1589_couleur': 'btv1b10534945n',
    'cuivre_isaac_langelier_paris1617': 'bpt6k722055',
    'cuivre_mathieu_langelier_paris1619': 'btv1b22000826',
    'cuivre_monconet_sommaville_paris1660': 'bpt6k87045023',
    'cuivre_tempesta_dejode_anvers1606': 'btv1b54000051z',
    'cuivre_tempesta_jansonius_amsterdam1610': 'bsb00008186',
}

PAT_ARK = re.compile(r'ark:/12148/([a-zA-Z0-9]+)|(bsb\d+)')


def extraire_ark(row):
    for col in ['version numérisée 1', 'version numérisée 2', 'url catalogue']:
        val = str(row.get(col))
        m = PAT_ARK.search(val)
        if m:
            return m.group(1) or m.group(2)
    return None


def normaliser(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]', '', s.lower())


xl = pd.ExcelFile(RACINE / "retours_celine" / "BNU_corpus.ods", engine='odf')
synthese = xl.parse('Synthèse', header=0)
synthese['ark_num'] = synthese.apply(extraire_ark, axis=1)
synthese['graveur_norm'] = synthese['graveur\xa0: Nom, Prénom'].apply(normaliser)
synthese['ville_norm'] = synthese['ville'].apply(normaliser)

dossiers = sorted(d.name for d in SEG_DIR.iterdir() if d.is_dir() and d.name != "bnu_corpus_celine")
print(f"{len(dossiers)} dossiers d'edition a rapprocher (bnu_corpus_celine traite a part)")

27 dossiers d'edition a rapprocher (bnu_corpus_celine traite a part)


In [10]:
def rapprocher(dossier):
    m = re.match(r"^(bois|cuivre)_([a-z0-9]+)_([a-z0-9]+)_([a-z]+?)(\d{4})(?:_(.+))?$", dossier)
    if not m:
        return None, "NOM NON PARSE"
    technique, graveur_court, editeur_court, ville_court, annee, variante = m.groups()

    ark = ARK_CONNU.get(dossier)
    if ark:
        candidats = synthese[synthese["ark_num"] == ark]
        if len(candidats):
            return candidats.iloc[0], "ark connu -> trouve"

    prefixe = "ark connu mais absent de Synthese" if ark else "pas d'ark"
    candidats = synthese[
        (synthese["ville_norm"].str.contains(ville_court, na=False))
        & (synthese["année"].astype(str).str.contains(str(annee), na=False))
    ]
    if len(candidats) == 0:
        return None, f"{prefixe} -> AUCUN CANDIDAT"
    if len(candidats) == 1:
        return candidats.iloc[0], f"{prefixe} -> trouve par ville+annee"
    affine = candidats[candidats["graveur_norm"].str.contains(graveur_court[:5], na=False)]
    if len(affine):
        return affine.iloc[0], f"{prefixe} -> affine par graveur ({len(candidats)} candidats)"
    return None, f"{prefixe} -> AMBIGU ({len(candidats)} candidats)"


CHAMPS_SYNTHESE = {
    "titre": "titre abrégé", "ville": "ville", "publisher": "publisher", "annee": "année",
    "langue": "langue", "technique": "technique", "graveur": "graveur\xa0: Nom, Prénom",
    "type_iconographique": "type iconographique", "cadre_grave": "cadre gravé",
    "famille_iconographique": "familles iconographiques", "ark_synthese": "ark_num",
}

meta_par_dossier = {}
for dossier in dossiers:
    ligne, statut = rapprocher(dossier)
    rec = {"statut": statut}
    if ligne is not None:
        for champ, col in CHAMPS_SYNTHESE.items():
            rec[champ] = ligne[col]
    meta_par_dossier[dossier] = rec

table_meta = pd.DataFrame(meta_par_dossier).T
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 30)
print(table_meta[["statut", "titre", "ville", "annee", "graveur", "famille_iconographique"]])
print()
print(table_meta["statut"].str.replace(r"\(.*\)", "", regex=True).value_counts())

                                                       statut                          titre                  ville      annee                        graveur famille_iconographique
bois_eskrich_rouille_lyon1556             ark connu -> trouve  Trois Premiers livres de l...                   Lyon       1556                Eskrich, Pierre                      7
bois_leroy_gueynard_lyon1510    ark connu mais absent de S...  P. Ovidii Nasonis Metamorp...                   Lyon       1510            Leroy II, Guillaume                      2
bois_salomon_rouille_lyon1557             ark connu -> trouve        La Metamorphose figurée                   Lyon       1557               Salomon, Bernard                      7
bois_solis_feyerabend_franc...            ark connu -> trouve  P. Ovidii Metamorphosis, O...  Francfort-sur-le-Main       1581                  Solis, Virgil                      7
bois_wickram_behem_mayence1545            ark connu -> trouve  P. Ovidii Nasonis dess all...   

### 2. Themes deja connus (73 illustrations, voir ci-dessus)

Reutilises tels quels — pas de re-classification, juste report par chemin de
fichier, pour ne pas perdre l'information deja validee.

In [11]:
# base_ovide (73 illustrations) construit plus haut dans ce meme notebook --
# reutilise directement en memoire ici (aussi persistee a part, mais uniquement
# pour ../comparaison_ovide_bibles/, sans rapport avec ce calcul).
THEME_CONNU = dict(zip(base_ovide["chemin"].apply(lambda c: Path(c).name), base_ovide["theme"]))
print(f"{len(THEME_CONNU)} themes deja connus (par nom de fichier)")

73 themes deja connus (par nom de fichier)


### 3. Vectorisation SigLIP (niveaux de gris) — tous les crops des 28 dossiers

Meme pretraitement que ci-dessus (niveaux de gris puis re-RGB, pour neutraliser
le coloriage) — la nouvelle base reste comparable
aux donnees existantes (`bibles_siglip.pkl`, et les 73 illustrations Ovide
deja themees construites plus haut).

In [12]:
processor, model_siglip = charger_siglip(DEVICE)
print("SigLIP charge")


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

SigLIP charge


In [13]:
import time

lignes = []
t0 = time.time()
total = 0
for dossier in dossiers + ["bnu_corpus_celine"]:
    meta = meta_par_dossier.get(dossier, {})
    fichiers = sorted((SEG_DIR / dossier).glob("*.jpg"))
    for k, chemin in enumerate(fichiers, 1):
        vec = embed_image(chemin, processor, model_siglip, DEVICE)
        lignes.append({
            "chemin": str(chemin),
            "dossier": dossier,
            "theme": THEME_CONNU.get(chemin.name),
            "titre": meta.get("titre"),
            "ville": meta.get("ville"),
            "publisher": meta.get("publisher"),
            "annee": meta.get("annee"),
            "langue": meta.get("langue"),
            "technique": meta.get("technique"),
            "graveur": meta.get("graveur"),
            "type_iconographique": meta.get("type_iconographique"),
            "cadre_grave": meta.get("cadre_grave"),
            "famille_iconographique": meta.get("famille_iconographique"),
            "ark": meta.get("ark_synthese"),
            "embedding": vec.tolist(),
        })
    total += len(fichiers)
    print(f"  {dossier:45s} {len(fichiers):4d} crops   (cumul {total}, {time.time()-t0:.0f}s)")

print(f"\nTermine : {len(lignes)} illustrations vectorisees en {time.time()-t0:.0f}s")

  bois_eskrich_rouille_lyon1556                   42 crops   (cumul 42, 1s)


  bois_leroy_gueynard_lyon1510                    19 crops   (cumul 61, 2s)


  bois_salomon_rouille_lyon1557                  161 crops   (cumul 222, 7s)


  bois_solis_feyerabend_francfort1581            184 crops   (cumul 406, 17s)


  bois_wickram_behem_mayence1545                  50 crops   (cumul 456, 20s)


...

  cuivre_philippe_hackiana_leyde1670              16 crops   (cumul 1820, 90s)


  cuivre_savery_farnaby_paris1637                 18 crops   (cumul 1838, 92s)


  cuivre_tempesta_dejode_anvers1606              139 crops   (cumul 1977, 100s)


  cuivre_tempesta_jansonius_amsterdam1610        149 crops   (cumul 2126, 109s)
  cuivre_weyen_barbin_paris1669                    3 crops   (cumul 2129, 109s)


  bnu_corpus_celine                               62 crops   (cumul 2191, 111s)

Termine : 2191 illustrations vectorisees en 111s


### 4. Sauvegarde

In [14]:
corpus_complet = pd.DataFrame(lignes)
print(corpus_complet.shape)
print(corpus_complet["theme"].notna().sum(), "illustrations avec theme connu")
print(corpus_complet.groupby("dossier").size().sort_values(ascending=False))

DOSSIER_VECTOR_DB = RACINE / "data" / "vector_bases"
chemin_sortie = DOSSIER_VECTOR_DB / "ovide_corpus_complet_siglip.pkl"
corpus_complet.to_pickle(chemin_sortie)
print(f"\nSauvegarde : {chemin_sortie}  ({chemin_sortie.stat().st_size / 1024:.0f} Ko)")

(2191, 15)
77 illustrations avec theme connu
dossier
bois_solis_feyerabend_francfort1581             184
cuivre_borcht_plantin_anvers1591                182
bois_salomon_rouille_lyon1557                   161
cuivre_baur_sn_augsbourg1709                    161
cuivre_tempesta_jansonius_amsterdam1610         149
cuivre_monconet_sommaville_paris1660            148
cuivre_tempesta_dejode_anvers1606               139
cuivre_depasse_jansonius_arnhem1607             136
cuivre_mathieu_langelier_paris1619              135
cuivre_depasse_depasse_koln1602                 134
cuivre_bouche_blaeu_amsterdam1702               126
cuivre_baur_sn_vienne1639                       125
bnu_corpus_celine                                62
bois_wickram_behem_mayence1545                   50
bois_eskrich_rouille_lyon1556                    42
cuivre_goltzius_goltzius_haarlem1589_couleur     38
cuivre_goltzius_goltzius_haarlem1589             38
cuivre_briot_drobet_lyon1628                     28
bois_leroy_


Sauvegarde : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/vector_bases/ovide_corpus_complet_siglip.pkl  (15278 Ko)
